<a href="https://colab.research.google.com/github/arturbernardo/word_vectors/blob/main/llm_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers torch sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

embedding_matrix = model.get_input_embeddings().weight.detach()

print("Shape da matriz de embeddings:", tuple(embedding_matrix.shape))
print("Tamanho do vocabulário:", embedding_matrix.shape[0])
print("Dimensão dos embeddings:", embedding_matrix.shape[1])

In [ ]:
def nearest_to_vector(vec, topn=10):
    vec = vec / vec.norm()
    emb_norm = embedding_matrix / embedding_matrix.norm(dim=1, keepdim=True)
    sims = torch.mv(emb_norm, vec)

    top_ids = torch.topk(sims, k=topn).indices.tolist()

    for idx in top_ids:
        tok = tokenizer.convert_ids_to_tokens([idx])[0]
        print(tok, float(sims[idx]))

def get_embedding(token_text):
    ids = tokenizer.encode(token_text, add_special_tokens=False)

    if len(ids) != 1:
        raise ValueError(f"'{token_text}' virou vários tokens: {tokenizer.convert_ids_to_tokens(ids)}")

    return embedding_matrix[ids[0]]

# Proximidade de palavras

eval de (king - man + woman) = queen com 0.70 de peso

eval de (king) = queen com 0.66 de peso

Distanciar mais king de man da um resultado mais próximo de rainha,
mostrando a relação geométrica e como ela acaba revelando significados.

In [ ]:
king = get_embedding(" king")
man = get_embedding(" man")
woman = get_embedding(" woman")

# analogia vetorial
v = king - man + woman

# buscar tokens mais próximos
nearest_to_vector(v, topn=15)

In [ ]:
nearest_to_vector(king, topn=15)

# Próximo token de uma frase

In [ ]:
text = "The king lives in the"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

next_token_logits = logits[0, -1, :]

probs = torch.softmax(next_token_logits, dim=-1)

next_token_id = torch.argmax(probs).item()
next_token = tokenizer.decode([next_token_id])

print("Frase:", text)
print("Próximo token mais provável:", repr(next_token))
print("Probabilidade:", probs[next_token_id].item())

In [ ]:
top_k = 10
top_probs, top_ids = torch.topk(probs, k=top_k)

print(f"Top {top_k} próximos tokens:")
for rank, (token_id, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
    token_str = tokenizer.decode([token_id])
    print(f"{rank:2d}. {repr(token_str):15s}  prob={prob:.6f}")